# Random Forest: Predicting Diabetes

**Goal:** predict whether a person has diabetes (`Diabetes_binary`: 0 = no, 1 = yes) from health and lifestyle answers in a 2015 CDC survey.

This notebook uses a **random forest**: a different kind of model that works like a team of simple decision-makers voting.

1. **Split** the data into a training set and a test set
2. **Build** the model inputs (`X`) and the answer we want to predict (`y`)
3. **Set class weights** so the model doesn't ignore the smaller group
4. **Train** the random forest
5. **See** which columns the forest relied on
6. **Predict** on the test set
7. **Evaluate** with accuracy, precision, and recall

## Load the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("diabetes_binary_health_indicators_BRFSS2015.csv")

# List of variables that are logically binary/categorical (factor in R)
binary_vars = [
    "Diabetes_binary", "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "DiffWalk", "Sex",
]

# Convert those columns to pandas 'category' dtype (equivalent of as.factor)
for col in binary_vars:
    df[col] = df[col].astype("category")

print(df.head())
print(df.dtypes)

## Step 1: Split into training and test sets

A model should be judged on data it has **never seen**. 

So we split the data:
- **Training set (80%)**: the model learns from this.
- **Test set (20%)**: we hide it and use it at the end to check the model.

In [ ]:
diabetic_train, diabetic_test = train_test_split(
    df, test_size=0.2, stratify=df["Diabetes_binary"], random_state=1
)

## Step 2: Build the model inputs (`X`) and the answer (`y`)

- **`X`**: the columns the model uses to make a prediction (the *predictors*)
- **`y`**: the answer we want it to predict

In [ ]:
# Build the design (model) matrix for the chosen predictors
predictors = [
    "HighBP", "HighChol", "HvyAlcoholConsump", "Stroke", "DiffWalk",
    "Age", "Income", "CholCheck", "GenHlth", "PhysActivity",
]

X_train = pd.get_dummies(diabetic_train[predictors], drop_first=True)
y_train = diabetic_train["Diabetes_binary"].astype(int)

X_test = pd.get_dummies(diabetic_test[predictors], drop_first=True)
# Make sure the test columns match the training columns exactly
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
y_test = diabetic_test["Diabetes_binary"].astype(int)

## Step 3: Class weights

Most people in the survey do **not** have diabetes. If the model sees mostly "no" answers, it can score well by almost always guessing "no" while missing the people we care about most.

**Class weights** fix this: each person in the smaller group counts for more during training, so both groups matter equally.

In [ ]:
class_counts = y_train.value_counts()
inv_weights = 1 / class_counts
sample_weights = y_train.map(inv_weights)

How it works: the weight for each class is `1 / (number of people in that class)`. The smaller group gets the bigger weight. Let's see:

## Step 4: Train the random forest

A **decision tree** is like a flowchart of yes/no questions: "Is the person's blood pressure high? Is their general health poor?" and so on, ending in a guess.

A **random forest** builds many trees (here, 100). Each tree learns from a slightly different random slice of the data, so they make different mistakes. Then the trees **vote**, and the share of trees that vote "diabetes" becomes the probability.

Settings we use:
- `n_estimators=100`: the number of trees
- `max_depth=8`: each tree can ask at most 8 questions in a row, which keeps trees simple so they don't just memorize the training data
- `n_jobs=-1`: use all your computer's cores

Unlike logistic regression, a forest doesn't need the columns scaled. We pass in the class weights from Step 3, so the smaller group still counts more. This step can take a minute.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,   # number of trees
    max_depth=8,        # each tree asks at most 8 questions in a row
    random_state=1,
    n_jobs=-1,          # use all CPU cores
)
model.fit(X_train, y_train, sample_weight=sample_weights)

print("Model trained!")

## Step 5: Which columns did the forest rely on most?

**Feature importance** shows how much each column helped the trees make their decisions. A longer bar means the forest used that column more.

!! Unlike lasso coefficients, importance does not tell you the *direction*. It shows which columns mattered, not whether they raise or lower the chance of diabetes.

In [ ]:
importance = pd.Series(model.feature_importances_, index=X_train.columns).sort_values()

importance.plot(kind="barh", figsize=(8, 5), title="Random forest feature importance")
plt.show()

## Step 6: Make predictions

The model gives a **probability** for each person in the test set. We turn that into a yes/no prediction using a **cutoff**: 0.55 or higher means we predict "diabetes".

We use the same cutoff as the logistic regression notebook so the models can be compared fairly.

In [ ]:
pred_probs = model.predict_proba(X_test)[:, 1]   # probability of diabetes
pred_class = (pred_probs >= 0.55).astype(int)    # 1 = diabetes, 0 = no diabetes

# Look at the first 10 predictions next to the real answers
pd.DataFrame({
    "probability": pred_probs[:10].round(3),
    "predicted": pred_class[:10],
    "actual": y_test.values[:10],
})

## Step 7: Accuracy vs. Precision vs. Recall

- **Accuracy**: out of everyone, how many did the model get right?
- **Precision**: of the people the model flagged as having diabetes, how many really do?
- **Recall**: of the people who really have diabetes, how many did the model catch?

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

accuracy = accuracy_score(y_test, pred_class)
precision = precision_score(y_test, pred_class)
recall = recall_score(y_test, pred_class)

print("Accuracy vs. Precision vs. Recall")
print(f"  Accuracy : {accuracy:.3f}")
print(f"  Precision: {precision:.3f}")
print(f"  Recall   : {recall:.3f}")

**How to read these:** accuracy alone can look fine even if the model misses many people with diabetes, so check **recall** too. Precision tells you how many of the model's "diabetes" flags are correct.

**Compare:** in the logistic regression notebook (same 10 columns, same split and cutoff) the scores were Accuracy 0.748, Precision 0.317, Recall 0.698. Did the forest do better, worse, or about the same?

## Recap

| Step | What we did | Why |
|------|-------------|-----|
| 1 | Split into training (80%) and test (20%) | Judge the model on data it hasn't seen |
| 2 | Built `X` (inputs) and `y` (answers) | Models need numbers in a consistent layout |
| 3 | Made class weights | Keep the model from ignoring the smaller group |
| 4 | Trained a random forest of 100 trees | Many simple trees voting can find patterns |
| 5 | Looked at feature importance | See which columns the forest used most |
| 6 | Predicted probabilities, then yes/no at a 0.55 cutoff | Turn the output into answers we can check |
| 7 | Printed accuracy, precision, and recall | See how well the model does on the test set |